# Đề 12 – Kaggle T4 v3: DINOv3 + SAM 3 image ckpt sam3.pt (fp16), sanity 5 frames, region / re-rank top-20 (proposal cache), 70 + 10 video. Built with DINOv3.
Trọng số offline từ dataset private `holilihu/de7-weights`. Toàn bộ kết quả → 1 file `/kaggle/working/de12_results.json`; dữ liệu làm việc ở /tmp.

In [ ]:
import os, sys, io, base64, tarfile, json, time, subprocess, traceback, glob
os.environ.update(HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1', DE12_FP16='1')
OUT='/kaggle/working/de12_results.json'; W0='/tmp/de12'; os.makedirs(W0, exist_ok=True); os.chdir(W0)
R={'started':time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime()), 'precision':'fp16 autocast (DINOv3 + SAM 3.1)', 'stages':{}, 'errors':[]}
def save():
    json.dump(R, open(OUT+'.tmp','w'), indent=1); os.replace(OUT+'.tmp', OUT)
def run(name, cmd, timeout=3000):
    t=time.time()
    try:
        p=subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout, cwd=W0)
        R['stages'][name]={'rc':p.returncode,'sec':round(time.time()-t,1),'tail':(p.stdout+p.stderr)[-3000:]}
        if p.returncode: R['errors'].append(name)
    except subprocess.TimeoutExpired as e:
        R['stages'][name]={'rc':'timeout','sec':round(time.time()-t,1)}; R['errors'].append(name+':timeout')
    save()
CODE = 'H4sIAFBq32oC/+1963bbRtJgfvMpepDMGJBIkOBNMhXmi2+JvblYkZxk5tD8MCAJkrBIAAJAShpZe/Zp9sH2SbYu3biRtJ2M45lvRJwjkWz0tbq6uqq6qtqsm/WvT53r564zcaPP/pCnwc+uz0aj1c6+Y7rVaFrNz8T1Z5/gWcWJE0Hzn93Pp3kslom3dPvW0cPmUfe402ma7Vaz024dVz7bP//xzzp2nWg8r/+RbeCiPup08NM66jTyn+matzrNbrcLqR1a/0fNo89E51Ou/1HwbnLzvvf/Qx/z34L+tzbpv7Wn/5+E/h/l6X/nuGt1zM5R+/jI2pP/+0T/bdvzvcS2zfDmD1n/3Xb7A+h/x+q0cP23Gs3WZ6LxKdf/nv7v+f97zv/Dqms2Gw/Nh9bxw4cP2/sN4B7R/3EQuX8E7X8//W9a7ZZF9P+o22m3G01Y/80mJO3p/yd4NE37xZu4gfCWzsytB6M37jgRjBQ9MY2cpRuL2ldCn64Wixr9Focicmde4BvCXY7cycTzZ5Tnm0cvzs/xS+zOlq6fxCbUXvGWYRAl4k0c+FURxFUxWwSjqkCio96N182q8FfL8EY4sfDDqpg6XgxZkwC6UZlGwVKcvvheyOwvsKucmkSOH0+DaOlGsXr9aJUEPwQTd1Glr6dRMHbjOIj4JxVO0yqVX148ffbSfvbXV6IvdM1chm2tKjTTWXv0uQzW/HnBn1fuaKkZlcrnolZ6UmCVnsrEnQpn7sRzfRoZvYqAZwaNwaDNyI29f7g6fh2vkyfBIoggU5XePXn5/csz+/G3Z81vzx79zagK/bgqjuHT8xM3CoOFk8Ac9DHrix9fPTuzH509e2RQ9ZGbrCJf6DPxlZiZS9fxdcMwp1AiceFrhboUO8tw4drcaz10kjmAPYz7ltmoiok7WYX9huwuTOPPvodg5lIw4SZn+arRE/GFFwqHhy+u5kHsim67NvIS4TyHUQsvFldeMvd88Xcq83cB72IRTEUyd8XCiRNx4YYJV0Dt6YCRjhgFURRcuRNBM/3Um/0cGqb4m+cuJrHQEzt2x1VEC5Om1CBkw9JUY1/8GPhc29gJJbgJ0584IUDHpREbJ2INY8bXTmjO3ISm4smjU/v07OWp/c3puSGCSDQ7ZoOqihMXkNOD/EvnWreqIgpW/kSnOuoIPQPmh7Nezb2FK15FK5dhiE9wASCOZGsRcFy6kb7zpsIPEsjSEyN4dZF/4Yk/U9Oi3xeNrDp85lBbilyYl0As3AXMQgqBXFX82vEnDCaYG2wVc1Ii4Jauz8Wf+vTaMOPVElBHfNnngsW2qUpx2BfWiRgHfuL5q2J7cibmhcQbnEAoVyfQV3k1mzjHThQ5N+9bDGffPjZyUKPm1TQDTBeuE7v69gXKVGvHCp1F3sSWOXRvOauKhbsGMPZ1oE0tmNZg7UYLBxaF2exk6+KpmwDBDCIgja4rWxBhFIRB7CzinioVwpIRy9Ui8Wrx2AHEwOZMcUYLNRYLDwAFC0K/blRvGtVrq3pjZQj9a1U8BzBCp0wkFyewMq5dRNrBkN7DwhQ+TJ3scTZJ4yuAHmLIrwBtH2uBjxMRQnKIyeMrcaB6SDnTX2kVWHeEdQOhnbm6bxRRAF+Pd7/GhzprAghcWCk6rpsG0TB9fDC+qoVXuGRyqdHBeF4L55Tq+fqvnKqPDy0D8h/K/PDmuXwT0Zv5IZYxCvSPWq5UxoCGsXj56/fr5hnPbzp7L0Okoc5CBNC72joYOyO5BfqwPWTzKPRvT3+GasfBEjY2WApAiGZBMFvAjnm1WDdrI8C6GlCU8dzq1lw/ho1x4aYTiPilBGw9dhdT2O2A1vW199eh4SYYXvSPkSavvTEUGocrLQfnnfvgS6w1t/vR72+C6CUNkBEXRp9WhB0zkVQWC9LStEOAaeR4vjvRsetAOCn7UmXfrHd7OTMJdB6IYbprZ5GjgFQhUEj6hDxQNQ5dDZzyfU0cgekH9ixKySeDF5bVIgUvrJUciK6hKh6dTmxO3KcFzmhiw54Iw4z7Wpho1D/Vgaxr0yXtITxkk+qwmfdxIz30rt2FDWNZQcXXAy3/Wxsag8YwrQf4HthlYflNoDas1IznTggreuo6SazSgCvAVB1zH2DmrCOAnFk/MkxFME88pEM61QSE25stAw8gVGhe0g1ZHn6WC1apB1hIiM/F+Hp8czXH5R06MNRJLb5cOZELxD6IYA8eNKpWVvm53BUVmQIUifJUSpGLN1gfdN3EyWVYXxim50M3gE4kAdJCvURHxtdAnm6qYgTQGCHl0mkogzdDIFnnRlqqUAhoqQBiKoCaihtL9g5+j69ro6t6MyM745vaaM4JQFdU/wEGmPdQ5c2/svDVzSGVKu+w11btugGMl9WlHfXGqt3wzx7CIyWESLuuG8zO6Tfqy7WlUiwjt81JghYhOduysSlU3NzZmPade8jqPfX8jBlOSeArXFBrL4YVy3uYe7nyAHmBf8cd6emLH1+uW794r4p8M645B7azmAYFIBHNZvsa/oC3A/YQxAe5sf/oJgIZ0HqcTN5FD7GOPpTPTTyhBtXdF7zvMd5CbZDihybzC4OG2T7uANtltjtd+mh0h4o+QaulvM3mQ8wETfFHZ/gOIoK0YjudKBIXaADEyPGFPtCxrZhb85YmsEWwoSa6BmwLEBfJ7+vp0KrZKA3FCj1+8f2LH589OgO+qw6sJ/CeopYNHRPToeF68pa4oLivww2UuS2Sox6LVCaPRb82aeOATc7FhdCqClgriO2T5CZ0+5x3ugicpNU07tRW+kyiW4pD44UX9mgPdbw6/qitvSS3n7Wa4i1M4QzeNHtq45S/i9seYtBbARJlsIacU2fsjoLgos4JtXgJ08MSAqW0hGuCJFLM1sLGY6hLbT21xXpidY8fLpHbnTmJO+kJ33WBgD3/BsBx4fog0S6A+sCWK5zxGGQRxx/nBIqd2/e7RqylGzYy12W0xvKIN/Bxkt/wuAjKHLo2Xk0cDSkKTwP+NL3YdtaOt3CANdANZvOZISjWj/tyKgfv3onTnW77XgxEGerREKzUEV199ann5gLEs0gnESl7A1IUyJmRLjcZWAFTDzZbbYldsRGvUJDWDNX79ULLyzq5llXTxb0giW56Ga+yKdJvH22hCvcaJxmWG/JJz0DKjHI1biGWMETYDZkUIlMoaSAjUZIRUNiWIsKngJjPrFEcaanTH9LfD+d2mFCN4j6s05y8uUo2918vY9cbKOT4kicysHxp393GOOHHwOt5h6N4uJ02nlC5W5Bk1wUko+aBn1tjD65NL3GXsQ5kpbSBvhcDBOkT5LJwAJZjkDPletlCuizYFWCdwoqZ9PVsrUHdvMJwlw5i0/XXXgRsK2oBtKfPrCaI/1YXcdXSDMoOn1vkm3nKUL2PEzRRHrbnHtBO34b9InE3KvNBgoT6dq0hf7UkGTVO3Mgm0hVDBxu0Ahs5HJ07sWgLlZOJXLzR2JQ4bKItTqIP5oMe1AVzSl8sIIrYm96QFUiWAW9qFq2EJ9+fw1vaiInaFRcX4vrWtuRoYHA2s9DIcq4i1D69G3CbWgdWlsCi82Mi1vpUKgvNV4SJRo9aBMQNggXAClZCuEoK9UCSYsbkMgORZeWPWSKERRfBboO79ZTRSMdt0Vv2AQYmUFzdMEljqW+yacACwIwBTAHrEleHhgzgCRAtdU1upjtUiID07vV2/cRo5S0mNmXQ16jHsiceMFlQOX8h8vreLSlV8ElNB4igkTeBF4tg1g8j4DzLuj9YGUvnAoSEKNbTxtxrQCw7uOijfgvWOzCgAG3FF+jUFwYLzJRSu+ASip1lSyvJrvIIwsR3SgOjZNjzRz+0pNB+QuI4EugsTZLEIsrJbHmBX4qcfegnU6JSt0gA13hD0vOaoHJGglamXePm1+4YiO8S1iwwBoiIDaK6VfprsKrbxH+yu0gGiQYCogI3ok8pZUqb5yIYmfhPB7ijhtJ8E4DgkZtw7QC3TujWVG2/QLkmMVJEPdVl5+kU1p2QRExNFrS+a8IIOeVlmQvAiMPCsUBhHM6tRh0BLlL1DtEL9yt9DatDS+ANa0QT1plpSMYhUaP2kJbCtgi/EXZ3wzLhx4nbQjmg/yOSQeG1XhTsN/qrVjRKapigjwxAz6VKzgawBPFuIH8NZdc5LaHfquM8+VnPR3dFaoRzr2oH7NKpF9QmoAMqJ5cngBOZlvKvLC3k6QNWAQUmuFfnUAWY/qSR4gvthwAEPQOl7FoOHN4kLkg7Hm/4IAJjURfolRthi9g3wiEgtTTQISI3fhuq/VMDUtBtawX9NPEKk9hgNfSmyteb4L5PRzfmCyRUL57+4IR6LuGbhZO8ONX/ymoPEKMRUlDMdCYTG1HYhvr1vw7gP/QE28rQmGq5ioBrkFQQygH6areSKt3Vb3EEdybllD3HXRZhcsvsJ04ynwxpgPfwi7BfQhJ/y2+QRksCUuizKjEYs+AK19ak9MWfCBSCJ6RSv/BcIl0HE1gIKNLL4nYIe1H+vT+tY5am5ILwkMycwLai3x4cUN+RcYbasePwcVclKUvPjxrTTSyIOHGlye0I6LlOFZAahrYl+rl122ECvFN7QG+VuLdFGOJh88Ygt4CtQg8KEzRC2AMnuhpIWjo/FCNVMOIykqzDZMCwGBar5c0n1YEAxqcbUVqMJn9Ykm8YX5FPZeTCExmJW8V+3V6kWMWLccdKxEWiKCNtkfHuikCaVuBk8GeaS97M+2n9F32rQWpgv99sNHCniYB5mjkhQbkqLgEbywAH2RUrkUe4IFPLo4m3IJhA79KTBxAEYHNPT2yBxhNNrOKpeFIFqlYNXefCTqoxWghUgQjeDVPBGJ9LACF2AMfOn7kjJdoo1STptI3AxI5cOha6vSvsUSmFG2wAdoiV43BoD8YByD14gInD0obwtCpeKJSBKR4QXTMlkC9Zk4fArJaz+EmQOAvD2DjhgFVIWst/eKH+lPSCL+D/O2UBwNPBm+GJuHBvUF05z+01c9pjtjK1mJsYW5+BhG2Lr+j7AN4NodVe9gsrZtYUhTeoFreoXMUzUlqn2EJAzH6Up8kyO0q1BCsGyeOwzNhyaYUrOnakiq0Yg97FkE+Vd2ZJlUU/YA4Bsh8weCDE3ghn8sYZo7pxjsfCeoz4ykgo3k6St1/2sTgqRYMUTU/UN0FoybpdLM+/UwQd3djrDNNwKpHlSfAwHcVlQwFayaI9KgG4kkj9JmYfgOSmtLYwGCoodxd3VjqGg+zzmGrFelStKRznsYk8nw6LYLyK8ofUOU4NG8Aq5nGvzCFhIRRXE2AO4PtAc2nz/rKPIN5Ex1wWYC22IRxiF+UiuGmAXfDLXIUTZBMorQ/dYRLQh74BhvVHxgfIe9zZHoFIQQ8SjG19pP1ZMWYAQY2ID+oq4btLfBh+417IH9zfHgIr5cxy1H1Lo2oXZJ4bs1RxvfUXznI0ccR1T9SuUzgY98X4bW//ubf/zNl/djsPW+Zx++jhUXNv/3mf7D8jFznBP8YC9D32/82j7pG0/+x2Gm2y/+zCgtzbf34a+0/WkcH+WEMUQEMlNM0Lwtp3Kb/VI97EdUBIHAP/4SF7oF6iLrGGwpxA9g03aZY5qqjFFtKgNLWkqejnj34QLTwh+fb05xOyg8IfT05/RttP1OZOap5vSFFStokaPdb5z11BDKLrj4H1B4kT2aHIZTaQ3krG0IkreLwfRN7M850FHfbbWJE9DmIPZKWcDOTWmCvIxKBD1MSgmWPOghVtV5FUVjfsVXdYqBJDTEIlAURq8djSMpFc4Q7jxCwrsopoJ1K2TTx9eW7/cP7sCVQlDoQF6whykoFhwb7wpGQXl+OE3m14t9XujgTci5x1YcV+8ujJ82fEY+PJgC4Zd+bVDDQGTqf+hCcTeC6WST1lYOUiBoxdKBSp0zUFO8RIPZYKiKrI6UFJTpblIyUxk2gIbB1I8SAq45dglZD0XJaRSUeb07adfJAgy/3ICbN5MSB9LUVNFs6xY9wj2YlLaMu3sedYKNPwkdiAVeXV45Ajh0C7dMJxKl0a9ENyysOC9g4qgxFhp7ao7qSYGufEVFZQ5WpDlW6hxpy0ymjQk59KOlXTg7apG0ZIuawZQA77pFCjPBu2sZS6pfPP8sCnuZlGSvubqY+pNBmGeEts/5n4WlyCQH0i0KQKbW7whelEM6QbxZHiGzQ2+gqhm4pKcVFQkqI4ZzVYViKzJGWrZFQlRrsTPjhJsYelwndII3kBBvKDvI2YDVNyq3GVUqtY1h+j2KTWH+ouGcx3lT3/v+f/9/z//kn5//zh68eWAt7D/3daVqfE/7e6R0d7/v8T8f/EkJvWhvMCG70H/uKGPA6IRY9WvrgCnvflyx9qF95iAayb/r+PzKb49rE4Oz/H0zw/IQt52ni/c2azhYucPrDbP6K5VE/MvESMF8hUKbs9YN8zJBR/+YsIvVCQZcdiIWquMKuV59/Yr15+9+xHlgLQSC9Gy6vM9I+K1ou/TcsUj05fEF8ZuWEgzp49evrDM8G2THwSCJz+OWTNDFuB83evSWBZholhVn6OIW+vYH5hmmbOgIJMGojTZc4GhCfst7JMiUHwSMZzFKtQPOF6UZKYg8QkD60mKEZI8PMxNoodO4QKeSiXGUHsPJeTbfV1jaUwrWpk3gQg3djMtzTMVsFWET1B3PFFCExmQiJI+WyJOkYAZps+go0bqX4yqIigsJkP5dpV2sxlzEzrlPlHfmoyhoxfktXOiTKZ/Ci2k6mkd1KGAQpVpRRoqGw2hrNiP/nu9JWWGSBH3hhPu7ZmPX919uLJKzzpaqQ2ZmlnGD/7O+Cp591B0H6HpIRgaT//pl/uqeT7S8efVMTzp0HxLA4198XyPZTsroLoImZBBecuTEjy5mVmk2dTuHCvIR2YUOfCFQ8m0jPKPHiAkgLZURN0l1683LAaG1+kxmh0OlzqAorvob0Ixuz0KAd95XqzeRLbSKOUKRJVNL5gIPO5LyD0RfHgIZ7QwS/IxeECKIaupZ0l21Q8s++JdcFeEaqUxzkk+ph0RsFmN7nSJWtGlN8YywnYZPNno7+FHk9AUiPc6H8DtLZkoLo5PxpAI9lmeFMClEGmC9wdG8EuTRTiCb4B0MdAiWRaZMrflBHfr3z3OoSy7iTNkiWluVQptiVCW4ZCRYNeZ1ioK5+xVB3mLYKMjNB4eRBS97RqCRxoKwMLY4yuQpOtxqNy0RGCFooOUgiA+BY5HlCAs5WP/C9ZAetTbleWp+aFLNG/LVX1QL54MLzLURC1aHnW3+PshPQODdryVE4az52kOXi7KDhHETnKdhJykkrp+TtconpS58MOs72tBJV9tFHGhm1w4Y0yuh1a3cxgdAtB/Q12tJWCIa/+4Za8bPmGPSHynXXT9FeLhfypG4ayBoWZciMXtmEimmW3olSiBpgSsMi8LAMgJW1zKKOFnNpGw9Shfo6pM1Wxw/UpZBVPNq+9skXqRp04Hptzkz2P26f/anPvh5vWCqMq65LQWgGqJLMANKNVZqyptxRZj0olw7b3Rm/HkXFfoWHRpQlxcQULnXybpLk30c6RYZDd8PXN9Y0g215gLtdu5E1vyBgd9oUNI9rBiMrb8jBcHttG7I8DZWOXKf5A2tVfDId7gfV/6vnvPv7bv0z/c1SM/9ltdM2H7WOrsw//ea/0P5KV/kMOgN+n/2lb3TT+p9U6wvNfPBLe638+jf7n5XSKLrtKmmIBXWpuJk4Cggaa5fkBuktOgisfWeDYMMX5KkS2EZmYGryrxckNlJgGqAuAAuy0ROa84lAcgJw/daWbGPKKN1dz4MzEykfNQf2Cmqt7Pnnp1ATGqhFTbwFcFEo5E/HgS+KLv7LtLzH5qwdCJ9/POfAqC2++qk/co5ocASljIrfmxBxDYcLGg6oRlGSBb69f1WWddbH2HBHfLAEMFzE2H4cL1FKlolUM7WM/vgIZK0rsH3+E5g9FoKJHqJfx3Gl2umZynTygPuSs/Hf3gYqC3FxRx9EoHzJjLJ1c6yRmWzaKgPKH+o5+PuiDNY/ZLhAD6gSzmYvhXW7Qwyl3fs1xl5C751hMGDUGOGfgFuerxFsgX8XHrhMPWEJgsXSajrivFaYH5W7ofZaajqXoRsT8HcKEGTVmJnlO+4qfCzO+OHN9IZtpbPmufnBQP9CwZ+NVFHtryfPl7ay9GKvUQ3lgxzz0rcbDB5FzgLbyMm/kLvBTxxg+1IBxJ/T0LQgs5J0dGvXmwUGz0TOt6Z344bGhZZ2k7g8zc9IFBTNALlo2CHIlmt5rQiA0jA0XKgRW2X+KMn3OOL+B55WC8EDtZ2z5iBVLRY1AWDi61Gyb3HJHRWZ+yYYCI5NQXcdcqPU44cgU2i32865+u7zTTgr9n2ztfDmI0YYN/ATqmoKcTAoMudJwGnKvJBQ2lh6l47KLi5a82+CxpPNyk9RLeqTp5qHBK1Z/PTk0vtCq26BVANeyx23lbYDTc3eAgCwD8rY5i4JVqFsobOZMhMPMxwuBRc68IK3FHEUDa96wCp6QOXwGdCxzBxRBq7wbrlCuJKIR1SDHignaXWtXIxC5nVhcbXerCnOSVRjD5DAlMMdBeIOQDUZv2E0DHWAiqItil1jiyy9Fs23k3YxxlVG3iTDZkXNFNsfQi0xmnqMjR2GVT9RgJeU8KPobYQmkahklKY3W8Umzij2El2iSL61eJFZjHBNSBoWQJL3lTshIX9I+k9stBQ4pQzGSUJzucE5bkD8ITGuk8/F9T0y5IwpUIMRqiPxzZTEAZYyy7J/BkHsFS4yAqM/NuXs98WYuCuWoQcGBw0CQ0KgyOHdYSMC6vKXKHpQryxRV82lGgW83kfv9NDm3t2+hzofFcozVxUKbduA44Tu8H7ctP63AUGiGIf3yQo8UwNKM4uACQz1cwxLm8yhyMIHRe3RGo5Nb3CQNY4A5KOkC13SmMcd9VM4b78h9bIWDIaxbqNniiBMa+WEX3pHPMH6zR9sKjbBQlXZx+Rq/pqWa+TJN5dpFAxyEJSJIDpSZK6iuIQHBiIVhMoeSQ5iVQVGjjcMqqLTTwlq2io2hAsFAy7gQwksGbK4f0DFyl1UGUBGVGJj28L+st1T4bXpQsJ0Uq6nYgD83/t6231Fl6n6McClULxVO0Mo9FHn39j97+5+i/c9Ds3kM3/f3v9yLJx4Dd5TE/xb3v7D9zxHd/3Jk7e9/2ev/9/T/E+v/O22rZXaOm0edh3vyf5/oPwbgmHggKzsU2OKjngO8R//fApyT+v9u96jdhfXf7uzvf/lk+v/vHXTIH7n+eL50Ioza5iyyn0K3Gux3E1fF5cqNPJA1V/ASI8hNUNjH0/gb8fTRLy/ORRNmVhxQJLsDEbuQHx17nJh8w4Irv5LhGFcq9G6DdOVHqhVTPM3yxMEqQhtKNEggHbps5/zlz2dPnp2bywkeNIQevEsCFcQxLmDycmKUvbhYE67U3uTNxSrxyK3E0ZjsIqriKcU+dBKnPnHWXmwTSFCkLqfZC4Rg+oZ6qFXyWlOO13JX5xFqGzrULfGfoAh0RRWpH9DNBFLdJ3W0uSrrtxui79SQutuCytCQZotKB6veO6OYVONTg+INY4cusUMDTc45Dk9+tedONNGGm325vfxdTcqRYums9TfbwaG6cCD1SAWdpf6Gps6oAObFSjMIJZ/e1cnk6dwFSocoWicMxYMarawyPBHQznJSLJ2hW5r/BLGNTXxy3luXUhm93pyhDCft2/iOprOyW7e7NkpuYFJdV4AI9Ox/nT77lsYW19vHjbB+iwo68004oxA6V1HeufDXiHSU62o5yZ7iMhvrBxr0ao2TgNFx9eMO/IdacxryTOcUlYJTLGVL3pLgE1LrHK9JxxC1gN5LGfap16R4UzpUXRXQiLTFzV2I4S0LzWftQ5VFX0ap8Vcap6n2+kDcQoIbjzGEdmzcvY4PXw++B8R7PXyt64P/NoaHxmsDrQpxnnkelfJ+qmHw2fI8ibcCvuH/TOlPZwXc7wex626QpQeQXzMqhEMajApjJdd30CcZuEkCS/s8TwAZeYSeI6+Eu8Zr/5EPeEP2sHFPPHkiHv9NtM2GyhpEM8cHWEZ4VPsNRZ/qyVjxczSSZrdYSWBVM5LMjhQ51/8WrF6tRm791EWjrTrizDqo/7//83/Jl42G4Y1WhTJ0Y4C06AcccMUkIATPZ1ek/rX/2n8rN4K32X7xdqN/byFfrVZL/177Guw8Gnywihgga8gEo8ImpGi+Wlgt+cUoyalBaliiycb95Hf3+r+9/i93/+fDo+Om2bIaXavV3QuA90j+y/k3fXQbsPfIf1az0VH3f7ZY/mt19/e/fZpHSkUgwIROFON9BOipgw5Q8EVa2ADLEiUYLLx8ELvrt+LsbRuPJm2bLuYpxPqlcB0FfzFCvQpFw1CdMR9FsxXG4zjFX3R274QUKtSRL3StVktlqghvroiUb/3WvAGZLxUzVrbkU35D0gLkPWGVt7eFEfSqglwYyLA+q80yG1tbVWFIc+3K6MzjeYC3o/QHmh/4JGnKFzJycVVGVh5u7wrHMpWdkTGeuQGY1Lm7CPsaX9U2d5ZL9lTEO2iCxaQqGv1gOqUh4syEJs0M1o7Bl/NekY6ptASOSfKzY8pYq45JYVYdM42uyiGmHZPjDu83mj3/t+f//vX8X3r+2209bLX3F8DfJ/4P3RNXTuL+C+5/7za7zdL5b8vq7O3/P5X+X6pUSaHawyD/5Meo9QQ7XmszMqNWEVqrMjprlSKz3g3vhqY44yDSdIulDDicxVIzK4/SoGwehjlYAK7h96nwEnXfZAzsxbev+GpbPH7QD+s1kQQLg1RHN3iJGDJbGG6CgohBXpUQ0F1NlTNyLo/FmYtem1/fWtVO1WrcCd3xb3KN+hzZDg14H51+bVE0bcdXEdAXaK0/vsmfF2ScMR8RSP5YhYLLRYH7g1jmcgSIE7GTk1aR2n8PG01M3Idx0dmJwIdx0gWOlmMDMidYpXht2xuByd/BPjfMzm7ufvsbjP3nRK6zmyFXfDCGElBBC9VpF9kDL534QmAdGN2QLokRXwKf7MU7muSwXPmRa6oN5N3JHrKngi4SUjYb3FI5Dov2Hrkna2AHl/4THlNcivRMpxSD3zHlQDm+3yV7buNQMezcmC8Y+hLZeQnF4Qld9qiwDSog5MHI7iAW5VwSdiF5rkXlKX17VxQQGXwKsflXBQMfqqDZaMKrcqU3s0gfk13rI3+zy4mQteUTK+6iXG/uxprfdluNrL54W81Pv1TFT6/S62Gyc7afuAm8paUvvfRpcgqG0AjeqrgcSAo9NIzSBYMnIimHNfzpF3W4cZ6LW4gvXqXXHxWDxvH5G4Xbh46xFBcrP5GqOoM8w0mzejSQDn9YDfy8OxGPTtn5/QQJqvx2+l0xUvqlV+WRZ9eS/JQ78yqNI+8WoKanVwqZgQG9bToOoqiR5yhxKpGU49rglKiIjBgeMg0M+dMvg0tvWLqvbaN6nM1twR13VgOjVzD+6RW+4riiRWCrzKvYnSBFXVAMAHLPL4UP2BIhUh2BsT10et/LrAhYwJgZhoPku8RlyEZsj9wrZmnQR0T4LAQkx88YyP2eo8yrcPKHAFug0CoLJ35FGVT2GmcpmFpnPv4LBZhlPs4moCV2C2mdviyeOpKPxVlPnA0uhulxHQUsgA1ehwrxDoL8UWES8JW9s5iCPJ5+t1mqOHIoUBz7rDB2eq1iYRaAgd0jiCs453pBdxr0RQNJOCJnQ94kn97GV+XrtbPJgqEYG4H/ox7XxJetY1XwjVLqAqb9UFhZk7D61FAxY50uvMDTMNU9xNj0puyY79DA1XxwcDvVzr6+vbhL76dhWPkh3wG3xgOzVvEev7P0Hj+MDEM81Y7S0C8uj9ctBaFlEzA93/721Y4Sp9+lBZYxHf3fpDkx4u1BMTssN8xP9+34tuJSOJDNTzIyD58N8rpNB29UNshKntYXdyFk9uwoOhE+et8NsPJ8rAt4ZXJoU5Pvr9OlCwW2lgbrxKt4yJHhVlv5HnTVVmFxVYf9kcFXG7ixTfeS4GUHq2UhHW8TUuVSwMhMOO/OtW5VVW0G3+yqpQ3xhUscHITrtjIXGBgc3g9VgFqx99VtI5KQBE6sl7/MiG9GglbOzQmeo6s7jegeMLrnhQ7I6QKmWF1yJNWIeDTOdwz5wKntlYX/ifq/vf3vv0z/ty3+R3cf/+O+6f9m4YqNGf8ABeD77H9brU4p/ker1dif/34q/R9GeQX+KV5FLl9+oMvgH6fAYwEb8apt9NTFwhywVB4RBqvZPMQtmq90SAP7SwUGiCRKUxKHLgg3Fboc4pczyI0BN3KR5FCzsHSXQXRjOwuK8ghCgCDjRNQYsvVchqLEWYgz11mg/m2E4UZQb5NX27GyjnV00tw3Vdxhu5VPdrytLiOsFu9erWB0EhV0b1s8PWRi6YJxmB+twtfD5zLjLcocW9CmzuH1DySTawAnurdyTXw2FMC48FTQtjFwGkyIbd9V1nxb6IZd6Q7z5rLlHEZurPBQ+IbYjAWmmmXwNjT93HLrLB6Bg1B2NgBGXbK9yA0jq8o/DYpEggiDt7JJF115t+y78UZH1rd5cNBqIMNbyeVGq8SEpQ5ZiLhW1JK4+euLN4IYAu7/0rJPH716jiftaYxhduWusfN3Td3gXlusJ1b3+OES2VYOZYgNyHENeu1hoVPxDYAYmkKz1236G1WQbnfvtjHLjsIVgKZ0Npf+6CxgeMsZ3zuaigg5MKP8WFZJSBGKALWGbPZshKcBNBd3FYxv6RZ64S7D5MYeO+M5D+EDIF75Lbq0ygZQoEBZtUbIVHQSf2d0XpU7DehJRSApC/FZysQoKnGwCDIOnQJZ1Sw3ftskkyiJEhiFasVqDCO7FFpO1btnPosPLKedxcOibLgx1TD9OWSQsnYmXBZLQy9L2ZvvwBOWGVMhMNbPclKcUYwqkwqAm1eLZ1LkmZQLte0bgrYXFff2H3v575+w/wD5r9Fog/x31D1qN/b3f9wn+Q9JMfl1Kb3pR5QD32f/2+1ayv633eqg/NdpNPf3/30q+e85zDrem1449O7xZW3Sl4hf0TV8Xizm04UXhnhaheEg47GzYPfLTuPPwKVGGIPRx/s5rhvmEV/QFzrkVwSSoFOJICFY1vwAo683m+1r+MNbBdcOSJ4gggZrkCjiseun4b2FQ/wHOy5RV4xUOFT+gJxOGMzi4YYsiKLfe+/sq/LHS7RXpW/P/Dne4lEZbXUGPRGRjwe2fmjysEx5GG9DOolkP229836UOTJyx7Nb7zmA4WBY9h/NFSH3y60+pMWjZHWSLMdgPk6nRlejNJceBb7PnThzW7eXgwfETVLUsNIxs2EAY0116jDFhmzOTOYgjPsgwOq6ZTWBJ7aaOKJRdqSdXTYI8DHR2mcGAinK3s1Opyr0Jvod0j9kRDn+O8B2BTmP5YnVaGYSNrGPIJYARhYav/ImyRz4WZD9uCJOnlNIUE7HzuCZYHaAnl0AIWFbL7u51rW02dhZuwo64Z3Gc6VO2m4PDi6haGo6dWdUMs6VzjF2zn2GtJss7InInNnQs/Wj87T/Hvxfc5P/a+z5v0/C/3UL/l/H3XbLfNhtdo/23N+94v9CoIRO5Nq0u33kM4D32f+2rdT/y+p02xj/wzrax//4VPzfY3TiEY6Il3hvGal5a6jL9NxiIBBimHJ+6GSni0EHkMdLXdEzT3TDrNTEkywEdiz+XquhVoma+Hvq7h1zaGy8EQ6d7XXYuqchB/HOMVysfoYaX5b4VOJKdTLPHOE1dYdW58/AcGLYiSkzsbJ3yIByWTRrQ1vMCZqMsnpN6FNgtBLsIvo9Qfrfpd4Ntn9VURbQJHLFjy9fCfTYh8Eh84vMqbI2i/knGayI9LJjHwOl0NZ+7U4IOOfE5aYjwTLTFcxBsWuY89tXvdQ4Gtjk2B0bZMs8d4tBxtmVnkOrZ+wyG6+SnZVfI22tqOcypcMCXugm3m78nIUN59OUD7wC+3cYItN0581ac0FV3uXVV8yf59K3GianiLjVL661244Yg3xsL9PYXkji09YywCG/z3mwkB/DLm8z8S0FPHFMGv9G2BPAnq1BTwY9PkvBajYkD7LD2Rm7pqjIzbJnRuIlOSUAPjm+zExg8ZwnRW19unDQjjOYTqURHPHOscxOlr8507lp0XSOCmOgW7pSbZi/4WoKeclYMbsEGyv1yiG2c7mUUWIvz+fr0DNoAe3upFsjJ+geyBuWoZJR1IiVqXIpyCydk3l08RHGUELph9T+AHsDbVZTzJT9p5AjZMp4GQ/WXm/tkfllmm14ItZ4BEghZxjvbyFPvZ7P1Gs0Jxx0RkaFwZ6dUOczo0QyMnVJiKRGM/i9CXeEoEkxbTMQjXuZD0Xj+O+rIRfPpFhF6M+0nEklEjNCCJAMWZjMSa+hkQtf7fgAm8IxJxrWokBJNne0adBZDR6G3DbusjYui9Fs3oSY60Q8r4pfUYDMAtmkJTg3h3DZWAU8FJu0GgSRqrg0cpevhyjzMzY4pqQXjBVvQgSeL89GMWfuDrnLVPpMBc+ptrtJDVLQ0s+93PDq6TEKYR6yeO0hZsAvNHDtKRT3Cyh+N7wrWugGHgXoAngXzVfx3P06Zi0JbTop3NHqFkpt3CGAQ72O0emg3WhsufAdn2tYNtdWVdzA540FtV+jMsPHc3P8hle4w4s07UamgUB9XRXhjbzy3bQ64kDo1xYM7bphkOidpt5g6k0+/NGHTjRsu7cwsDs11wNsGzvaqIU3Rg/79By7fQi/qkK+vIaX1/zyVxzbIfwalq53IwIHqD8KgoUu4eilcDTxCNEQXwF/1uBlQGQGsWZYjDb/AZhTGkMBeTLnkPyR8jUfJz8H2P0Kw2ob1c0o7/y8F/viFPfcOyZLsI8Qdm3bKcqYmI66iIjTjUXtDTdwb5oLUvWnfJCqXlpeBqmabg+RJesBIpvuNFvDcJV31votAeFO+4DwXHLr2dV8GnhrmrukAdcwrEG1fCsb4bRy+qrLao6VyON3QV25RVuV1RHvrkPSpO1ntqzvgkLpnnhHOx+KA7fFmE5U7eGD1DLlgQF5+ccJ5w0uIaWo2K7ymxjfxHn++35GgNrb/+7tfwv6P6uD8X+Pj/bu//dK/0e6ij/k9r8PiP901Mj0fy1c/8320d7//5M8u+I/se3svXFpZyb4Qx3a89omcmHPR2himzlyakLTRI0M6iDPjthMF1v1PJ0drtzxdr/rsrcsxeAkw1vlr5oTkaEY3f9dOtRVAZvw80KqHlJmLBo8IMbqwfDudYK/iFF/MKQ78mqYANy6/BkLnQy9MZX9JdUL43VCl033qQb8hm9adPubakq63N2Sf1/eXrGWGD2zMb0Ty3sbrPM/lv/b2//9y/i/ov2f1T6yzKZ13O029vE/7xP/F618dKGw8cwPNo35x17/77j/x2o11f3PnSPr6IjOf7vtPf/3KZ7P/1QfeX595MTzyufiG4q5Tee+SyeJvGu+SZnN7TnkvjhBR4WlE90IvDWaYlKoHIhEU6zCXASzintNHNyrs0c/nn/z8uyHZ2fn9i/Pzh6/PH/x6m99F+3OKu5aN26FC9yT0Pp98YUlvmiKL1raiQhvkjl6jm3GJxOSs+PTYW9ybUOxNCqRKB//pWqnL5o8gC9ags4M025/cWvdwdsv2vy++dVfLPFWzCI3FLVnQvtv9nd4y/zcW2TmDOjgXWUgatOsEwVvoDo6uHN1Q/H2bXk0xWi7QgXz2ey7TOf+bm9KyHCl4kP9kgQdLwqEmfK5wRg4hXFTjcRG+nRHZqFBvo6yxvYCW0OinohJAAykuxZf+CJn5XZSSBK5yEx4BtXRhE2VQtEBZuuXGh6Kv/xF3BbqUDWSKg8nZYInbnu6vuf/9vzf7+D/Ou2jY/P4qN18uLcAvHf8H/n5fmTe70P4v3ZX8X/dIxn/s2V19/zfp+f/6CqwWuwmzAPq6aVcBrrbA9+Hruk9EV94wECQXQ+GAmCTNjSJYY6KL6Q2P4QBPBGP+9v9zXPc1fc2Mhm/k6X64vEWJoprTDmnd0V3z3FJGHu90C3mTj5ux7jO38PU5Zk5Yt1+olu+M+4rZZmIOaOBKBaYAWJ/8dPucezkwFNgKg4cRpby3LJCGQhUGkMW2e9i25s9kz7tv7NvKTzf2btif4otMktJMsqj779/+vLHZ/9BO+Oe/9vzfzn+r93udE2r2+m02nv+7z7xfxw5I1yZtO19/PX/zvjvzabi/zoNTLdaHWt//vtJnok3ncL+N/MS4RAS1InvqE/cMXxGuKGOtqdXeJttPnScScs1za41nkxGTWHRZFdqtdquGiuHh4c7a/36a1FrNRrVI3HIH5AwXjhxLF5Fjh8DX7N0o6ecX/d984dgslq4pYipFDLVjYPFisJP5Yy52bw7ARbJLdh4b1oqTl0nsdHQj0IKp3XV67L4ZolxEEST2J5X1bcrstpeTE0bozVxol6pbbOITFur5r9ygKc+RxGqHP7mgjqXRHDYNp+o27Yuo0IZ26JO8ZWWGlACzdgc4ZYkGt84WIZcA4wymnAsIBi8vgmSD6gihv65EwV7fdsIjcouxA2D2MOJskGUCDBS2QYKb8shkXnaOh53myPTPD7qum7L3YnMW+vYQOutuRDBO03Eb/qfovepzPxMhVg793x3N4bjo4MYgHBbJa5dRNFWsyre8bJsmVu0DmaDW4S+5+drwaS4t4m/ievHQRSn8cn+4UZBrGPoWQsv5dyCysYmLv+WSv45tN6FgTDqK5CSdNmTLdk+F9NVlMzdCKZM0ZIJCH8YIjhJ/bEI9/U3sLGgX9jIFbEzdXeiPeYe4OiGiljkkkxqRzdMbkXfjfZZ+Cc7jIKxG8MQNjB/RyaF/Mft1sNpxzSb4/Fk2pnuRP5d1Wzg/66MuASsNi4B+g8/+WJanL+1hwHqzEQR+zQg2LpZkSvlHKo9VfXJZaFpGv5VBKMnOhUBavgeIgZClWO1L6o5Ut6H8R2XEBPplD8F6u6PXTu9BY2unOgxzv72mv85bH1nj4q4xBqDvjy8Kb7LUYD8dlbKxT3Gcxf6shPb1gug88sREKhNJCu+k7g1bhxN3YeWaYKQMRl1H+7ErVLpDZQqvSdMahI15Y+UnGKwt1++f+yML0bBLjqaOWUBfQPUEWqCJSmwMfpIbsPm6R47YcLX2Hk+lOMI2n2+EMAByo0vnQWVlak7tvF/srqPsblvgQaG7FyTGxZF0rOvInQViXRmZPKQsf3ABvhSNkMvEjg1qn46PJ6r1hHNFX38prnK6smYvDK8sldbAZdNZHHNZzPycVbsJmgZw/iaE8EsbrNLLC59pID4hYjfS39x8x8IhFEsvdjUKIxtANprHfb6v73+71+l/7O6lmW2ug+73eO9/u8+6v+QUR995Fsg3+f/cdRqpfZ/bauF9n/Ndmev//sUD8hMpzjjMoY7CWso07rjBF26ffHk9OeewOB15GAAkjjwgjU+xpMJxHuQzoCC9QECLQGj8mFEOAw7eZZQ3BCOwL49YgilIjayOGPT8SnGTclfFp4TLKWlWrGUuU3yTF1K8uLjZmxpDJu3vRU95ZiQwzkRdL1bvjJ9meP1MQ8yTxqnaKgfKOQ2lRgJX1CdCV/Ggb12yIsmZmaqVL1yk9BkQOrq7nDUVTFdrOK5dGQZk/tM6lf8BDiwVeRuxnpXvrSc1GhwsPcTZDxNjJ2ANTx5dGrjTWr26ctz+4fzZ0/wMrUGBj8JLqrsJA25OcBIJX+hXRb9D6sZr5MnwQJGNWWH5ycvv395Zj/+9qx59u1jg31YKIrDQONza41iTUfSbdjxYWYW2pAZzM0I4Zhyin3mGdShI+goY5UzcrxFzklCDeOvzh/9EPAWYwf1Yxl4gScgJId4WXd+IqwM/hpXsXuaLHVXlcaxHqArA43cYmK6pWsgL3TCONvZBXCFXHjDmhdTBJfOcFic8/3mtn/2z/7ZP/tn/+yf/bN/9s/+2T/7Z//sn/2zf/bP/tk/+2f/3NPn/wM8GSGqABgBAA=='
tarfile.open(fileobj=io.BytesIO(base64.b64decode(CODE)), mode='r:gz').extractall('.')
sys.path.insert(0, W0); os.makedirs('results', exist_ok=True)
from vsearch.weights import discover
t=time.time(); Wt = discover('/kaggle/input', '/tmp/w'); Wt['assemble_s']=round(time.time()-t,1)
R['weights']={k:v for k,v in Wt.items() if k!='layout'}
ok = Wt.get('dinov3') and Wt.get('sam3_ckpt') and Wt.get('sam3_sha256_ok', False)
if not ok: R['errors'].append('weights missing / sha mismatch'); save(); raise SystemExit('stop')
os.environ['DINOV3_PATH']=Wt['dinov3']; os.environ['SAM3_CKPT']=Wt.get('sam3_ckpt') or ''; os.environ['SAM3_STRICT']='1'
run('nvidia_smi','nvidia-smi --query-gpu=name,memory.total --format=csv'); print(R['weights'])

In [ ]:
run('install','pip -q install faiss-cpu pycocotools psutil && git clone -q https://github.com/facebookresearch/sam3.git /tmp/sam3 && pip -q install -e /tmp/sam3 && cd /tmp/sam3 && git rev-parse HEAD', 900)
print(R['stages']['install'])

In [ ]:
run('data','wget -q https://data.vision.ee.ethz.ch/csergi/share/davis/DAVIS-2017-trainval-480p.zip -O /tmp/davis.zip && mkdir -p data && unzip -q /tmp/davis.zip -d data && rm /tmp/davis.zip && python scripts/prepare_davis.py && python scripts/make_hard_queries.py && python scripts/add_distractors.py', 1200)
print(R['stages']['data']['rc'], R['stages']['data'].get('sec'))

In [ ]:
T0=time.time(); BUDGET=52*60  # stop starting new stages after ~52 min (kernel cap 1h)
def left(): return BUDGET-(time.time()-T0)
SAN=["import json, glob, sys, os; sys.path.insert(0, '.')",
 "from vsearch.core import sample_frames",
 "from vsearch.sam3_regions import SAM3Regions",
 "fr = [im for v in sorted(glob.glob('data/davis_bench_large/videos/*.mp4'))[:5] for _, im in list(sample_frames(v, 1.0))[:1]]",
 "sam = SAM3Regions(min_score=0.3); n = [len(sam(f)) for f in fr]",
 "json.dump({'load_info': sam.load_info, 'boxes_per_frame': n, 'ok': sum(n) > 0}, open('results/sanity.json', 'w'), indent=1)"]
open('sanity.py','w').write(chr(10).join(SAN))
run('sam_sanity','python sanity.py', 600)
try: R['sam_sanity']=json.load(open('results/sanity.json'))
except Exception as e: R['sam_sanity']={'ok':False,'error':repr(e)}
SAM_OK=bool(R['sam_sanity'].get('ok')); save(); print(R['sam_sanity'])
if not SAM_OK: R['errors'].append('SAM sanity: 0 boxes or load failure -> SAM stages skipped')
run('gpu_bench','python scripts/gpu_bench.py', 900)
try: R['gpu_bench']=json.load(open('results/gpu_bench.json'))
except Exception as e: R['errors'].append('gpu_bench json: %r'%e)
save(); print(R.get('gpu_bench'))

In [ ]:
R['index']={}; R['eval']={}; R['skipped']=[]
SETS={'large70':'data/davis_bench_large','small10':'data/davis_bench'}; D=os.environ['DINOV3_PATH']
for sn,B in SETS.items():
    run(f'index_grid_{sn}',f'python scripts/build_index.py --videos {B}/videos --out data/{sn}_grid --model {D} --regions grid', 900)
    if SAM_OK and left()>600: run(f'index_sam3_{sn}',f'python scripts/build_index.py --videos {B}/videos --out data/{sn}_sam3 --model {D} --regions sam3', 1200)
    elif SAM_OK: R['skipped'].append(f'index_sam3_{sn} (time)')
for d in glob.glob('data/*_grid')+glob.glob('data/*_sam3'):
    try: m=json.load(open(d+'/meta.json')); m.pop('meta',None); R['index'][os.path.basename(d)]=m
    except Exception as e: R['errors'].append(f'index meta {d}: {e!r}')
save(); print(R['index'])

In [ ]:
for sn,B in SETS.items():
  for Q in ['object_hard','object']:
    jobs=[('dinov3_grid',f'--index data/{sn}_grid')]
    if SAM_OK: jobs+=[('sam3_region',f'--index data/{sn}_sam3'),('rerank_sam3_top20',f'--index data/{sn}_grid --modes both --rerank sam3 --videos {B}/videos')]
    for tag,cmd in jobs:
        if not os.path.exists(cmd.split()[1]+'/meta.json'): R['skipped'].append(f'{sn}/{tag}/{Q} (no index)'); continue
        if tag.startswith('rerank') and left()<300: R['skipped'].append(f'{sn}/{tag}/{Q} (time)'); continue
        o=f'results/E_{sn}_{tag}_{Q}.json'; run(f'eval_{sn}_{tag}_{Q}', f'python scripts/evaluate.py {cmd} --queries {B}/queries_{Q}.json --out {o}', max(120,int(left())))
        try: R['eval'][f'{sn}/{tag}/{Q}']=json.load(open(o))['results']
        except Exception as e: R['errors'].append(f'eval {sn}/{tag}/{Q}: {e!r}')
        save()
R['finished']=time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime()); save()
print(json.dumps(R['eval'], indent=1)[:4000]); print('errors', R['errors'], 'skipped', R['skipped'])
for f in os.listdir('/kaggle/working'):
    if f!='de12_results.json': subprocess.run(['rm','-rf','/kaggle/working/'+f])
print(os.listdir('/kaggle/working'))